In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300


font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 9}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 9}

matplotlib.rc('font', **font)

In [2]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(apply_inclusion_criteria=False)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3254: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,78

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
24-hour segments: 410
12-hour segments: 820

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [3]:
# summary_subjects_icu = pd.read_csv('summary_subjects_icu_1.csv')
# summary_days_icu = pd.read_csv('summary_days_icu_1.csv')

In [19]:
cohort = 'icu'
print(f'ALL SUBSEQUENT ANALYSIS IS DONE FOR COHORT = {cohort}')

if cohort == 'icu':
    data_subjects = summary_subjects_icu.query('inclusion_subject == 1').copy()
    data_days = summary_days_icu.query('inclusion_subject == 1').copy()
    days_max = 9
elif cohort == 'sleeplab':
    data_subjects = summary_subjects_sleeplab.query('matched_all == 1').copy()
    data_days = summary_days_sleeplab.query('matched_all == 1').copy()
    
    for col in data_subjects.columns[11:]: # add pseudo full day columns
        col_f = col.replace('n1_', 'f1_')
        data_subjects[col_f] = data_subjects[col]
        
    days_max = 1

print(data_subjects.shape)


ALL SUBSEQUENT ANALYSIS IS DONE FOR COHORT = icu
(102, 171872)


In [20]:
data_subjects.iloc[:, -10:].head()

,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_mean,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_std,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_median,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_q25,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_q75,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_sda,f9_hrv_sd1sd2_stagewise_disrelaxed_amendsumeffort_S_asd,airgo_available_h_sum,ecg_available_sum,airgo_ecg_available_sum
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.020250,48.000556,12.020250
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.497528,92.276444,35.103444
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.580028,58.004194,51.500861
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.197472,66.900778,41.032833
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.634444,82.764722,38.542194


In [21]:
data_night = data_days[data_days.day_cat == 'n'].copy()
data_day = data_days[data_days.day_cat == 'd'].copy()
data_full = data_days[data_days.day_cat == 'f'].copy()

if cohort == 'icu':
    data_full_mindata = data_full.query('airgo_ecg_available > 0')
elif cohort == 'sleeplab':
    data_full_mindata = data_night.query('airgo_ecg_available > 0')

In [22]:
print(f' Total hours of Airgo available:                {data_subjects.airgo_available_h_sum.sum() :.2f}')
print(f' Total hours of ECG available:                 {data_subjects.ecg_available_sum.sum() :.2f}')
print(f' Total hours of ECG and AirGo available:       {data_subjects.airgo_ecg_available_sum.sum() :.2f}') #todo

print('For all 24 hour segments that satisfy airgo_ecg_available > 0:')
print(f' Mean (STD) hours of Airgo available:           {data_full_mindata.airgo_available_h.mean():.2f} ({data_full_mindata.airgo_available_h.std():.2f}) ')
print(f' Mean (STD) hours of ECG available:            {data_full_mindata.ecg_available.mean():.2f} ({data_full_mindata.ecg_available.std():.2f}) ')
print(f' Mean (STD) hours of ECG and AirGo available:  {data_full_mindata.airgo_ecg_available.mean():.2f} ({data_full_mindata.airgo_ecg_available.std():.2f}) ')
print(f' Median (IQR) hours of Airgo available:           {data_full_mindata.airgo_available_h.median():.2f} ({data_full_mindata.airgo_available_h.quantile(0.25):.2f} - {data_full_mindata.airgo_available_h.quantile(0.75):.2f}) ')
print(f' Median (IQR) hours of ECG available:            {data_full_mindata.ecg_available.median():.2f} ({data_full_mindata.ecg_available.quantile(0.25):.2f} - {data_full_mindata.ecg_available.quantile(0.75):.2f})')
print(f' Median (IQR) hours of ECG and AirGo available:  {data_full_mindata.airgo_ecg_available.median():.2f} ({data_full_mindata.airgo_ecg_available.quantile(0.25):.2f} - {data_full_mindata.airgo_ecg_available.quantile(0.75):.2f})')

print('Subject Level:')
print(f' Mean (STD) hours of Airgo available:           {data_subjects.airgo_available_h_sum.mean():.2f} ({data_subjects.airgo_available_h_sum.std():.2f}) ')
print(f' Mean (STD) hours of ECG available:            {data_subjects.ecg_available_sum.mean():.2f} ({data_subjects.ecg_available_sum.std():.2f}) ')
print(f' Mean (STD) hours of ECG and AirGo available:  {data_subjects.airgo_ecg_available_sum.mean():.2f} ({data_subjects.airgo_ecg_available_sum.std():.2f}) ')
print(f' Median (IQR) hours of Airgo available:           {data_subjects.airgo_available_h_sum.median():.2f} ({data_subjects.airgo_available_h_sum.quantile(0.25):.2f} - {data_subjects.airgo_available_h_sum.quantile(0.75):.2f}) ')
print(f' Median (IQR) hours of ECG available:            {data_subjects.ecg_available_sum.median():.2f} ({data_subjects.ecg_available_sum.quantile(0.25):.2f} - {data_subjects.ecg_available_sum.quantile(0.75):.2f})')
print(f' Median (IQR) hours of ECG and AirGo available:  {data_subjects.airgo_ecg_available_sum.median():.2f} ({data_subjects.airgo_ecg_available_sum.quantile(0.25):.2f} - {data_subjects.airgo_ecg_available_sum.quantile(0.75):.2f})')

# print('')
# print(f' Total hours of sleep:             {data_full.sleep_hours_breathing.sum():.2f}')
# print(f' Total hours of sleep ASWTI:       {data_full.sleep_hours_breathing_aswti.sum():.2f}')
# print('For all 24 hour segments:')
# print(f' Mean (STD) hours of sleep:        {data_full.sleep_hours_breathing.mean():.2f} ({data_full.sleep_hours_breathing.std():.2f}) ')
# print(f' Mean (STD) hours of sleep ASWTI:  {data_full.sleep_hours_breathing_aswti.mean():.2f} ({data_full.sleep_hours_breathing_aswti.std():.2f}) ')
# print(f' Median (IQR) hours of sleep:      {data_full.sleep_hours_breathing.median():.2f} ({data_full.sleep_hours_breathing.quantile(0.25):.2f} - {data_full.sleep_hours_breathing.quantile(0.75):.2f}) ')
# print(f' Median (IQR) hours of sleep ASWTI:{data_full.sleep_hours_breathing_aswti.median():.2f} ({data_full.sleep_hours_breathing_aswti.quantile(0.25):.2f} - {data_full.sleep_hours_breathing_aswti.quantile(0.75):.2f})')


 Total hours of Airgo available:                3886.14
 Total hours of ECG available:                 6727.77
 Total hours of ECG and AirGo available:       3501.57
For all 24 hour segments that satisfy airgo_ecg_available > 0:
 Mean (STD) hours of Airgo available:           13.92 (7.15) 
 Mean (STD) hours of ECG available:            18.71 (6.95) 
 Mean (STD) hours of ECG and AirGo available:  12.83 (7.11) 
 Median (IQR) hours of Airgo available:           13.10 (9.17 - 21.32) 
 Median (IQR) hours of ECG available:            22.73 (13.01 - 23.84)
 Median (IQR) hours of ECG and AirGo available:  12.34 (7.83 - 18.63)
Subject Level:
 Mean (STD) hours of Airgo available:           38.10 (28.05) 
 Mean (STD) hours of ECG available:            65.96 (27.80) 
 Mean (STD) hours of ECG and AirGo available:  34.33 (24.57) 
 Median (IQR) hours of Airgo available:           31.27 (18.41 - 48.17) 
 Median (IQR) hours of ECG available:            61.76 (48.04 - 76.50)
 Median (IQR) hours of ECG a

In [13]:
plt.figure(figsize=(3,3))
# data_subjects.airgo_available_h_sum.hist(histtype='step', bins=np.arange(0,80, 1))
# data_subjects.ecg_available_sum.hist(histtype='step', bins=np.arange(0,80, 1))
data_subjects.airgo_ecg_available_sum.dropna().hist(histtype='step', bins=[0,2,10,20,30,40,50,60,70,80])
plt.xlabel('Hours of Data Available')
plt.ylabel('Number of Patients')
plt.tight_layout()
# plt.legend(['hours of airgo and ecg available'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
# development/old code:

In [5]:
# exclude patients with little data only.

# Patients without sufficient amount of sleep might then not be used in particular analysis but it seems wrong to exclude them
# from the beginning (here we are interested in general data statistics) as this sleep staging is part of our result.

In [32]:
# import re
# # add total amount of data availability

# for col in ['airgo_available_h', 'ecg_available', 'airgo_ecg_available']:
#     cols = [x for x in data_subjects if re.match(f'f\d_{col}', x) is not None]
#     data_subjects[col + '_sum'] = data_subjects[cols].sum(axis=1)

In [33]:
# min_hours_available = 2
# min_age = 45
# # min_data_columns = [f'f{i}_airgo_available_h' for i in range(1,9)]
# # min_sleep_columns = [f'f{i}_sleep_hours_breathing' for i in range(1,9)]

# print(f"N Patients Start: {data_subjects.shape[0]}")

# data_subjects_inclusion = data_subjects.loc[data_subjects.age >= min_age, :]
# print(f"N Patients After Age exclusion: {data_subjects_inclusion.shape[0]}")

# data_subjects_inclusion = data_subjects_inclusion.loc[data_subjects_inclusion.airgo_ecg_available_sum >= min_hours_available, :]
# print(f"N Patients After Little Data Available exclusion: {data_subjects_inclusion.shape[0]}")

In [10]:
data_subjects['inclusion'] = np.isin(data_subjects.study_id, data_subjects_inclusion.study_id).astype(int)
data_days['inclusion'] = np.isin(data_days.study_id, data_subjects_inclusion.study_id).astype(int)

In [12]:
print(data_subjects['inclusion'].sum())
print(data_days['inclusion'].sum())

102
1161


In [13]:
# min_hours_sleep = 0
# min_hours_available = 2
# min_age = 45
# # min_data_columns = [f'f{i}_airgo_available_h' for i in range(1,9)]
# # min_sleep_columns = [f'f{i}_sleep_hours_breathing' for i in range(1,9)]

# print(f"N Patients before min age exclusion: {data_days['study_id'].unique().shape[0]}")
# print(f"Number of Full Days:")
# print(data_days.shape[0])

# data_days = data_days.loc[data_days.age >= 45, :]
# data_subjects = data_subjects.loc[data_subjects.age >= 45, :]
# print('age exlusion done')

# print(f"N Patients before min data availability exclusion: {data_days['study_id'].unique().shape[0]}")
# print(f"Number of Full Days before min data availability criteria:")
# print(data_days.query(f"day_cat == 'f'").shape[0])

# # we remove all data where not 2 hours of data is available for both ecg and airgo.
# for i in range(1, days_max + 1):
    
#     min_data_available = data_subjects[f'f{i}_airgo_ecg_available'] >= min_hours_available
#     cols_day = [x for x in data_subjects.columns if f'f{i}' in x]
#     data_subjects.loc[~min_data_available, cols_day] = np.nan
    
# data_days = data_days[(data_days.airgo_ecg_available >= min_hours_available)]

# print(f"N Patients after min data availability exclusion: {data_days['study_id'].unique().shape[0]}")
# print(f"Number of Full Days after min data availability, where we do have ECG&AirGo together >= 2 hours available:")
# print(data_days.query(f"day_cat == 'f'").shape[0])

# if 0:
#     # we remove all data where the amount of sleep is not satisifed.
#     for i in range(1, days_max + 1):
#         min_sleep_fulfilled_breathing = data_subjects[f'f{i}_sleep_hours_breathing'] >= min_hours_sleep
#         min_sleep_fulfilled_ecg = data_subjects[f'f{i}_sleep_hours_ecg_nn'] >= min_hours_sleep

#         cols_day = [x for x in data_subjects.columns if f'f{i}' in x]
#         data_subjects.loc[~(min_sleep_fulfilled_breathing | min_sleep_fulfilled_ecg), cols_day] = np.nan

#     data_days = data_days[(data_days.sleep_hours_breathing >= min_hours_sleep) |
#                                 (data_days.sleep_hours_ecg_nn >= min_hours_sleep)]

#     print(f"N Patients after min sleep exclusion: {data_days['study_id'].unique().shape[0]}")
#     print(f"Number of Full Days after min sleep exclusion criteria, where we do have ECG&AirGo available:")
#     print(data_days.shape[0])

 Total hours of Airgo available:                4034.68
 Total hours of ECG available:                 7018.46
 Total hours of ECG and AirGo available:       3583.40
For all 24 hour segments:
 Mean (STD) hours of Airgo available:           9.84 (8.66) 
 Mean (STD) hours of ECG available:            17.12 (7.99) 
 Mean (STD) hours of ECG and AirGo available:  8.74 (8.34) 
 Median (IQR) hours of Airgo available:           10.25 (0.00 - 15.33) 
 Median (IQR) hours of ECG available:            21.73 (10.78 - 23.82)
 Median (IQR) hours of ECG and AirGo available:  8.39 (0.00 - 13.74)
Subject Level:
 Mean (STD) hours of Airgo available:           38.10 (28.05) 
 Mean (STD) hours of ECG available:            65.96 (27.80) 
 Mean (STD) hours of ECG and AirGo available:  34.33 (24.57) 
 Median (IQR) hours of Airgo available:           31.27 (18.41 - 48.17) 
 Median (IQR) hours of ECG available:            61.76 (48.04 - 76.50)
 Median (IQR) hours of ECG and AirGo available:  28.40 (16.23 - 44.7

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …